In [ ]:
import sys
import os
sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet")

In [ ]:
# Importing dependency
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras import datasets
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.layers import BatchNormalization
import numpy as np

import tensorflow as tf
import tensorflow.python.keras.callbacks as cb

import time
from silx.io.dictdump import dicttoh5
from tensorflow.python.util.tf_export import keras_export


from keras.distribute import distributed_file_utils


In [ ]:
def _set_device(device, index):
  _DEVICE = "CPU"
  if (device.upper() == "GPU"):
    _DEVICE = "GPU"
  _PHY_DEVICE = tf.config.list_physical_devices(_DEVICE)[index]
  if (_DEVICE == "GPU"):
    tf.config.set_visible_devices(_PHY_DEVICE, _DEVICE)
  else:
    tf.config.set_visible_devices(_PHY_DEVICE)
  _DEVICE += f":{index}"

_set_device("gpu", 1)

In [ ]:

@keras_export('keras.callbacks.GradientsTracker')
class GradientsTracker(cb.Callback):

  # Initialization
  # ===================================
  def __init__(
    self,
    data=None,
    max_samples=20000,
    freq=50,
    save=False,
    h5filename="res",
    log_dir=None,
    **kwargs
  ):
    super(GradientsTracker, self).__init__(**kwargs)
    # Dataset
    self.data = data
    self.max_samples = max_samples
    # Computing and saving/writing gradients
    self.freq = freq
    self.save = save
    self.h5filename = h5filename
    self.log_dir = log_dir

    self.histogram_freq = freq

  def set_model(self, model):
    super(GradientsTracker, self).set_model(model)
    self.weights_names = tf.nest.map_structure(
      lambda w: w.name.replace(":", "_"), self.model.trainable_weights
    )
    self._log_write_dir = distributed_file_utils.write_dirpath(
      self.log_dir, self.model.distribute_strategy
    )

    self._train_dir = os.path.join(self._log_write_dir, "train")
    self._grads_writer = tf.summary.create_file_writer(self._train_dir + "/gradients")

  # Callback methods
  # ===================================
  # 'on_train_begin' method
  # -----------------------------------
  def on_train_begin(self, logs={}):
    self.compute_gradients = self.make_compute_gradients()

  def make_compute_gradients(self):
    grads = K.gradients(
      self.model.total_loss, self.model.trainable_weights
    )
    inputs = self.model._feed_inputs \
      + self.model._feed_targets \
      + self.model._feed_sample_weights
    return K.function(inputs, grads)

  # 'on_epoch_end' method
  # -----------------------------------
  def on_epoch_end(self, epoch, logs={}):
    if (epoch % self.freq == 0):
      start = time.time()
      # Sample dataset
      data = list(self.data) + [None]
      # Compute gradients
      grads = self.compute_gradients(data)
      grads = {name: grads[i] for (i, name) in enumerate(self.weights_names)}
      # Compute gradients mean
      grads_mean = {name: np.mean(np.abs(grad)) for (name, grad) in grads.items()}
      # Store gradients
      self.store_grads(epoch, grads, grads_mean)
      end = time.time()
      print("\nTIME:",end - start)

  def store_grads(self, epoch, gradients, gradients_mean):
    # Save gradients to hdf5
    if self.save:
      data = {
        str(epoch).zfill(6): {
          "tensors": gradients,
          "mean": gradients_mean
        }
      }
      dicttoh5(treedict=data, h5file=self.h5filename, mode='a')
    # Write gradients to TensorBoard
    if (self.histogram_freq and (epoch % self.histogram_freq == 0)):
        self._log_grads(epoch, gradients)

  def _log_grads(self, epoch, gradients):
    with self._grads_writer.as_default():
      with tf.summary.record_if(True):
        for name in self.weights_names:
          tf.summary.histogram(
            name="grad_" + name + "/histogram",
            data=gradients[name],
            step=epoch
          )
          print(gradients[name])
        self._grads_writer.flush()


In [ ]:


tf.keras.backend.clear_session()  # For easy reset of notebook state.
tf.compat.v1.disable_eager_execution()

# Import Data
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Build Model
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(10))

# Model Summary
model.summary()

# Model Compile 
model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
# Define the Gradient Fucntion
epoch_gradient = []

# Define the Gradient Function
def get_gradient_func(model):
    grads = K.gradients(model.total_loss, model.trainable_weights)
    inputs = model._feed_inputs + model._feed_targets + model._feed_sample_weights
    func = K.function(inputs, grads)
    return func

# Define the Required Callback Function
class GradientCalcCallback(keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs=None):
      get_gradient = get_gradient_func(model)
      grads = get_gradient([train_images, train_labels, None])
    #   grads = dict(zip(wnames, grads))
      epoch_gradient.append(grads)

# callbacks = [GradientCalcCallback()]

In [ ]:
log_dir = "/home/zanardi/Codes/ML/PyCOMET/pycomet/pycomet/optimization/callbacks/test/logs/"
os.makedirs(log_dir, exist_ok=True)

gradtrack = GradientsTracker(
    data=tuple([train_images, train_labels]),
    max_samples=None,
    freq=1,
    save=False,
    h5filename=log_dir + "/gfile.h5",
    log_dir=log_dir
)

tboard = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=0,
    write_graph=False,
    write_images=False,
    profile_batch=0
)

callbacks = [gradtrack, tboard]

In [ ]:
epoch = 5

model.fit(train_images, train_labels, batch_size=2048, epochs=epoch, validation_data=(test_images, test_labels), callbacks=callbacks)


In [ ]:
epoch_gradient